# 00: Colab CPU Replay

This notebook runs on **CPU** and uses the DeepSeek API for:
- Generating debate traces
- Counterfactual replay
- Building datasets

**Persistence**: All data is stored on HF Hub.

## Setup

In [ ]:
# Install dependencies
!pip install -r requirements-api.txt

In [ ]:
# HF Hub setup
from huggingface_hub import snapshot_download, upload_folder

# Download data from HF Hub
snapshot_download(
    "YOURNAME/causal-mas-distill-data",
    repo_type="dataset",
    local_dir="data"
)

In [ ]:
# Import modules
import os
from src.backends.api import APIBackend
from src.debate.harness import DebateHarness

# Set API key
os.environ["DEEPSEEK_API_KEY"] = "your-api-key-here"

## Step 1: Select Hard Problems

In [ ]:
# Run problem selection
!python scripts/00_select_hard_problems.py \
    --input data/problems.json \
    --output data/hard_problems.json \
    --difficulty-threshold 0.5 \
    --max-problems 1000

## Step 2: Generate Debates

In [ ]:
!python scripts/01_generate_debates.py \
    --problems data/hard_problems.json \
    --output data/traces/debates.json \
    --api-url https://api.deepseek.com/v1 \
    --model deepseek-chat \
    --max-rounds 3

## Step 3: Counterfactual Replay

In [ ]:
!python scripts/02_counterfactual_replay.py \
    --traces data/traces/debates.json \
    --output data/counterfactuals/results.json \
    --api-url https://api.deepseek.com/v1 \
    --model deepseek-chat \
    --sample-size 100

## Step 4: Noise Floor Estimation

In [ ]:
!python scripts/02b_noise_floor.py \
    --traces data/traces/debates.json \
    --counterfactuals data/counterfactuals/results.json \
    --output data/noise_floor/results.json

## Step 5: Build Datasets

In [ ]:
!python scripts/03_build_datasets.py \
    --traces data/traces/debates.json \
    --utilities data/utilities.json \
    --output-dir data/datasets \
    --token-budget 100000

## Upload to HF Hub

In [ ]:
# Upload all data to HF Hub
upload_folder(
    folder_path="data",
    repo_id="YOURNAME/causal-mas-distill-data",
    repo_type="dataset"
)